#### Datos: mapa propuesta de la línea de enrgía (febrero 2026)

El siguiente script tiene como objetivo obtener la información sobre el 


por consumo de energia eléctrica en cada alcaldía de la Ciudad de México. 


Fuente: Censos Económicos 2024


Link: https://www.inegi.org.mx/app/saic/default.html


Link de base de datos: https://docs.google.com/spreadsheets/d/1bCl3Gg9EsDheWaaDr44toIWUFAazgKmABu-uquGKw_o/edit?gid=0#gid=0

Link de consulta de la clasificación de las actividades económicas: https://docs.google.com/spreadsheets/d/1ckOabnLFTHOPRR5vCYwbpKJJH4_MthNgUYycVm6sMP4/edit?gid=0#gid=0

In [2]:
#Se limpia el entorno de trabajo
rm(list=ls())

if (!require("pacman")) install.packages("pacman")
pacman::p_load(tidyverse, srvyr, openxlsx, readr, janitor, dplyr, stringr, googlesheets4)

In [8]:
# Configurar para que no pida autenticación
gs4_deauth()

# Usar la URL completa de tu hoja de cálculo
datos <- read_sheet("https://docs.google.com/spreadsheets/d/1bCl3Gg9EsDheWaaDr44toIWUFAazgKmABu-uquGKw_o/edit?gid=0#gid=0")

head(datos)

✔ Reading from censos_economicos.

✔ Range Hoja 1.



Año Censal,Entidad,Municipio,Actividad económica,UE Unidades económicas,K412A Gasto por consumo de energía eléctrica (millones de pesos)
<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>
2023,09 Ciudad de México,NA,Total estatal,425005,33463.106
2023,09 Ciudad de México,NA,"Sector 11 Agricultura, cría y explotación de animales, aprovechamiento forestal, pesca y caza",34,0.148
2023,09 Ciudad de México,NA,Sector 21 Minería,25,723.760
2023,09 Ciudad de México,NA,"Sector 22 Generación, transmisión, distribución y comercialización de energía eléctrica, suministro de agua y de gas natural por ductos al consumidor final",118,3696.517
2023,09 Ciudad de México,NA,Sector 23 Construcción,1358,310.278
2023,09 Ciudad de México,NA,Sector 31-33 Industrias manufactureras,31218,5224.887


In [14]:
datos <- datos %>% 
  clean_names() %>% 
  filter(
    actividad_economica != "Total estatal",
    actividad_economica != "Total municipal",
    !is.na(municipio),
    municipio != "",
    !is.na(k412a_gasto_por_consumo_de_energia_electrica_millones_de_pesos),
    actividad_economica != "Sector 11 Agricultura, cría y explotación de animales, aprovechamiento forestal, pesca y caza"
  ) %>% 
  mutate(
    sector = str_extract(actividad_economica, "\\d+(?:-\\d+)?"),
    gran_sector = case_when(
      sector %in% c("11", "21", "22", "23", "31-33") ~ "Industria",
      sector %in% c("43", "46") ~ "Comercio",
      sector %in% c("48-49", "51", "52", "53", "54",
                    "55", "56", "61", "62", "71",
                    "72", "81") ~ "Servicios",
      TRUE ~ NA_character_
    ),
    gasto_miles = k412a_gasto_por_consumo_de_energia_electrica_millones_de_pesos * 1000
  ) %>%
  rename(
    gasto_energia_millones = k412a_gasto_por_consumo_de_energia_electrica_millones_de_pesos
  )

head(datos)

ano_censal,entidad,municipio,actividad_economica,ue_unidades_economicas,gasto_energia_millones,sector,gran_sector,gasto_miles
<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>
2023,09 Ciudad de México,002 Azcapotzalco,Sector 23 Construcción,60,1.414,23,Industria,1414
2023,09 Ciudad de México,002 Azcapotzalco,Sector 31-33 Industrias manufactureras,1519,1143.090,31-33,Industria,1143090
2023,09 Ciudad de México,002 Azcapotzalco,Sector 43 Comercio al por mayor,721,335.125,43,Comercio,335125
2023,09 Ciudad de México,002 Azcapotzalco,Sector 46 Comercio al por menor,7430,246.202,46,Comercio,246202
2023,09 Ciudad de México,002 Azcapotzalco,"Sector 48-49 Transportes, correos y almacenamiento",139,67.254,48-49,Servicios,67254
2023,09 Ciudad de México,002 Azcapotzalco,Sector 51 Información en medios masivos,36,14.187,51,Servicios,14187


In [11]:
tabla_municipal <- datos %>% 
  group_by(municipio, gran_sector) %>% 
  summarise(
    gasto_total = sum(gasto_miles, na.rm = TRUE),
    .groups = "drop"
  ) %>% 
  group_by(municipio) %>% 
  mutate(
    total_municipal = sum(gasto_total),
    participacion = round((gasto_total / total_municipal)*100
  , 2)) %>% 
  ungroup()
tabla_municipal

municipio,gran_sector,gasto_total,total_municipal,participacion
<chr>,<chr>,<dbl>,<dbl>,<dbl>
002 Azcapotzalco,Comercio,581327,2039002,28.51
002 Azcapotzalco,Industria,1144504,2039002,56.13
002 Azcapotzalco,Servicios,313171,2039002,15.36
003 Coyoacán,Comercio,368995,1336251,27.61
003 Coyoacán,Industria,179536,1336251,13.44
003 Coyoacán,Servicios,787720,1336251,58.95
004 Cuajimalpa de Morelos,Comercio,389412,1010347,38.54
004 Cuajimalpa de Morelos,Industria,47470,1010347,4.70
004 Cuajimalpa de Morelos,Servicios,573465,1010347,56.76


In [17]:
promedio_energia <- datos %>% 
  group_by(municipio, gran_sector) %>% 
  summarise(
    gasto_total_miles = sum(gasto_energia_millones * 1000, na.rm = TRUE),
    total_ue = sum(ue_unidades_economicas, na.rm = TRUE),
    .groups = "drop"
  ) %>% 
  mutate(
    gasto_promedio_miles_por_ue = if_else(
      total_ue > 0,
      gasto_total_miles / total_ue,
      NA_real_
    )
  ) %>% 
  mutate(gasto_promedio_miles_por_ue = round(gasto_promedio_miles_por_ue, 2))
promedio_energia

municipio,gran_sector,gasto_total_miles,total_ue,gasto_promedio_miles_por_ue
<chr>,<chr>,<dbl>,<dbl>,<dbl>
002 Azcapotzalco,Comercio,581327,8151,71.32
002 Azcapotzalco,Industria,1144504,1579,724.83
002 Azcapotzalco,Servicios,313171,7368,42.50
003 Coyoacán,Comercio,368995,10070,36.64
003 Coyoacán,Industria,179536,1558,115.23
003 Coyoacán,Servicios,787720,11020,71.48
004 Cuajimalpa de Morelos,Comercio,389412,3925,99.21
004 Cuajimalpa de Morelos,Industria,47470,526,90.25
004 Cuajimalpa de Morelos,Servicios,573465,3551,161.49


In [18]:
promedio_energia_tot <- datos %>% 
  group_by(municipio) %>% 
  summarise(
    gasto_total_miles = sum(gasto_energia_millones * 1000, na.rm = TRUE),
    total_ue = sum(ue_unidades_economicas, na.rm = TRUE),
    .groups = "drop"
  ) %>% 
  mutate(
    gasto_promedio_miles_por_ue = if_else(
      total_ue > 0,
      gasto_total_miles / total_ue,
      NA_real_
    )
  ) %>% 
  mutate(gasto_promedio_miles_por_ue = round(gasto_promedio_miles_por_ue, 2))
promedio_energia_tot

municipio,gasto_total_miles,total_ue,gasto_promedio_miles_por_ue
<chr>,<dbl>,<dbl>,<dbl>
002 Azcapotzalco,2039002,17098,119.25
003 Coyoacán,1336251,22648,59.00
004 Cuajimalpa de Morelos,1010347,8002,126.26
005 Gustavo A. Madero,1210885,46809,25.87
006 Iztacalco,502485,16013,31.38
007 Iztapalapa,2800579,82011,34.15
008 La Magdalena Contreras,111436,7714,14.45
009 Milpa Alta,27684,6894,4.02
010 Álvaro Obregón,2045439,22136,92.40
